# Distillation without SOLD tweets in SemiSOLD

The public SemiSOLD release contains nearly every SOLD training and test tweet
(with different spacing and punctuation). The earlier distillation run used
them. This run removes every SemiSOLD row that matches a SOLD tweet, then
repeats the experiment with the same settings: word + BPE-1k model, sentence
head with lambda 0.3, cross-entropy, no CRF, batch 64, train on the
6,000-tweet training part, early stopping on validation. Distillation weights
0, 0.5 and 1.0. Five seeds instead of the earlier three.

## How to run

1. Settings: Accelerator = GPU T4 x2, Internet = On
2. Leave `SMOKE = True` and Run All (about 10 minutes). It projects the full
   runtime.
3. Set `SMOKE = False`, then Save Version -> Save & Run All (Commit). Collect
   `/kaggle/working/distill_nosold/` when it finishes.

## Settings

In [ ]:
SMOKE  = True
BRANCH = 'main'
REPO   = 'https://github.com/hatheem-r/project_DNN.git'
print('SMOKE TEST' if SMOKE else 'FULL RUN')

## GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

## Code

In [ ]:
import os, shutil, subprocess
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q -b $BRANCH $REPO project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
helptext = subprocess.run(['python', 'notebooks/09_pieces_234.py', '--help'],
                          capture_output=True, text=True).stdout
assert '--exclude-sold' in helptext, 'clone is stale'
print('HEAD:', subprocess.run(['git', 'log', '-1', '--oneline'],
                              capture_output=True, text=True).stdout.strip())

## Tests

The alignment tests check that piece lists line up with words. A break there
shifts labels against tokens without raising an error.

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## fastText vectors and the embedding matrix

The model keeps the word channel, as in the earlier run. About 460 MB,
downloaded to `/kaggle/temp` so it is not saved as output.

In [ ]:
os.makedirs('/kaggle/temp/embeddings', exist_ok=True)
VEC = '/kaggle/temp/embeddings/cc.si.300.vec.gz'
if not os.path.exists(VEC):
    !wget -q -O $VEC https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
os.environ['SOLD_VECTORS'] = VEC
!python notebooks/03_embeddings.py 2>&1 | tail -4
assert os.path.exists('artifacts/embedding_matrix.npy'), 'embedding matrix was not built'

## Run

The log and the results CSV are copied to the output folder when it finishes.

In [ ]:
OUT = '/kaggle/working/distill_nosold'
os.makedirs(OUT, exist_ok=True)
CSV = 'results/results_piece4_nosold.csv'
if os.path.exists(CSV):
    os.remove(CSV)

args = ['python', 'notebooks/09_pieces_234.py', '--piece4', '--lambda', '0.3',
        '--exclude-sold']
args += ['--seeds', '1', '--epochs', '2', '--patience', '2'] if SMOKE else \
        ['--seeds', '1', '2', '3', '4', '5']
LOG = 'results/piece4_nosold_report.txt'
print('$', ' '.join(args), flush=True)
with open(LOG, 'w') as log:
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        if 'it/s]' in line or 'B/s]' in line:
            continue
        print(line, end='', flush=True)
        log.write(line); log.flush()
    rc = p.wait()
print('exit code', rc)
assert rc == 0, f'failed - see {LOG}'
assert 'removed' in open(LOG).read(), 'SOLD tweets were not removed'
if not SMOKE:
    shutil.copy(LOG, OUT)
    shutil.copy(CSV, OUT)

## Results

The earlier run (with SOLD tweets in SemiSOLD, three seeds) and the model
without a sentence head are quoted for comparison. A difference larger than
twice the larger standard deviation is reported as clear, larger than one as
modest, and anything smaller as no measurable difference.

In [ ]:
import pandas as pd

df = pd.read_csv(CSV)
if SMOKE:
    print('SMOKE TEST PASSED - every step ran. Scores are meaningless.\n')
    total = 0
    for tag, g in df.groupby('tag', sort=False):
        est = g['seconds'].mean() / 2 * 50 * 5 / 60    # 2 smoke epochs -> ~50 epochs, 5 seeds
        total += est
        print(f'  {tag:<24} ~{est:4.0f} min')
    print(f'  {"TOTAL":<24} ~{total:4.0f} min  (~{total / 60:.1f} h)')
    print('\nNow set SMOKE = False and use Save Version -> Save & Run All.')
else:
    agg = df.groupby('tag').agg(seeds=('seed', 'count'),
                                val=('val_f1', 'mean'), sd=('val_f1', 'std'))
    print(agg.round(4).to_string(), '\n')

    REF = {   # validation F1: mean, sd, source
        'no sentence head (lambda 0)':           (0.7050, 0.0025, 'results/piece2_report.txt, 5 seeds'),
        'earlier run, weight 1.0, with overlap': (0.7086, 0.0032, 'results/piece4_report.txt, 3 seeds'),
    }
    for k, (m, s, src) in REF.items():
        print(f'  {k:<40} {m:.4f} +/- {s:.4f}   ({src})')
    print()

    def compare(question, a, b):
        d = a[0] - b[0]
        noise = max(a[1], b[1], 1e-4)
        if d > 2 * noise:    verdict = 'CLEARLY HIGHER'
        elif d > noise:      verdict = 'modestly higher'
        elif d > -noise:     verdict = 'NO MEASURABLE DIFFERENCE'
        elif d > -2 * noise: verdict = 'modestly lower'
        else:                verdict = 'CLEARLY LOWER'
        print(f'{question}\n  {a[0]:.4f} vs {b[0]:.4f}: {d:+.4f} (larger std {noise:.4f}) -> {verdict}\n')

    row = lambda t: (agg.loc[t, 'val'], agg.loc[t, 'sd'])
    ctrl = row('piece4_dw0.0_nosold')
    for w in ('0.5', '1.0'):
        compare(f'Distillation weight {w} vs the same model without distillation:',
                row(f'piece4_dw{w}_nosold'), ctrl)
    compare('Distillation weight 1.0 vs the model without a sentence head:',
            row('piece4_dw1.0_nosold'), REF['no sentence head (lambda 0)'][:2])
    compare('Weight 1.0 without the overlapping tweets vs the earlier run with them:',
            row('piece4_dw1.0_nosold'), REF['earlier run, weight 1.0, with overlap'][:2])
    shutil.copy(CSV, OUT)
    print('saved to', OUT, ':', sorted(os.listdir(OUT)))